# Fine-tune NER — NlpHUST/ner-vietnamese-electra-base

Notebook fine-tune mô hình **`NlpHUST/ner-vietnamese-electra-base`** trên bộ dữ liệu pháp lý (BIO tagging) với các kỹ thuật tối ưu:

- **Subword label alignment** (chỉ gán label cho first subtoken, còn lại `-100`)
- **Mixed precision (fp16)** + **gradient accumulation**
- **Layer-wise Learning Rate Decay (LLRD)** — kỹ thuật mạnh cho fine-tune transformer
- **Linear warmup + decay**, **weight decay**, **gradient clipping**
- **Label smoothing**
- **Early stopping** theo `eval_f1` (seqeval entity-level)
- **Dynamic padding** với `DataCollatorForTokenClassification`
- **Reproducible seed**, auto-discover label space, lưu `id2label` cùng model

Input:
- `dataset/ner_finetune_data.jsonl` (train)
- `dataset/ner_val.jsonl` (validation)
- `dataset/ner_test.jsonl` (test)

Output: `models/ner-legal-vi-electra/` (model + tokenizer + label mapping).

## 1. Cài đặt môi trường

In [1]:
# Chạy 1 lần nếu chưa cài. Bỏ comment nếu cần.
%pip install -q "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" "seqeval" "evaluate" "torch" "numpy" "pandas"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [2]:
import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

torch 2.10.0+cu128 | cuda True
device: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Cấu hình

In [4]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Paths ---
PROJECT_ROOT = Path("..").resolve()
DATA_DIR  = PROJECT_ROOT / "/content/dataset"
TRAIN_FILE = DATA_DIR / "ner_finetune_data.jsonl"
VAL_FILE   = DATA_DIR / "ner_val.jsonl"
TEST_FILE  = DATA_DIR / "ner_test.jsonl"

OUTPUT_DIR = PROJECT_ROOT / "models" / "ner-legal-vi-electra"
LOG_DIR    = PROJECT_ROOT / "models" / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- Model ---
BASE_MODEL = "NlpHUST/ner-vietnamese-electra-base"
MAX_LENGTH = 256  # câu trong dataset rất ngắn

# --- Training hyperparams (tối ưu cho GPU 96GB — H100 / A100-80G+) ---
NUM_EPOCHS        = 30       # trần; early stopping sẽ dừng sớm
BATCH_SIZE_TRAIN  = 64       # effective batch = 64 — sweet spot cho NER fine-tune
BATCH_SIZE_EVAL   = 128
GRAD_ACCUM        = 1        # không cần với GPU lớn
BASE_LR           = 3e-5     # backbone Electra
CLASSIFIER_LR     = 1e-4     # head reinit — LR cao hơn
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
LABEL_SMOOTHING   = 0.1
MAX_GRAD_NORM     = 1.0
LLRD_DECAY        = 0.9      # layer-wise LR decay (1.0 = tắt)
EARLY_STOP_PAT    = 5        # kiên nhẫn hơn vì tổng epoch lớn hơn

# --- Precision ---
# Ưu tiên bf16 (ổn định, không cần loss scaler) trên Ampere/Hopper.
HAS_CUDA = torch.cuda.is_available()
SUPPORTS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
USE_BF16 = SUPPORTS_BF16
USE_FP16 = HAS_CUDA and not SUPPORTS_BF16

# Bật TF32 cho matmul / cuDNN — tăng tốc ~1.5x không ảnh hưởng chất lượng.
if HAS_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print("Train:", TRAIN_FILE)
print("Val  :", VAL_FILE)
print("Test :", TEST_FILE)
print("Out  :", OUTPUT_DIR)
print(f"precision: bf16={USE_BF16}  fp16={USE_FP16}  tf32=on")

Train: /content/dataset/ner_finetune_data.jsonl
Val  : /content/dataset/ner_val.jsonl
Test : /content/dataset/ner_test.jsonl
Out  : /models/ner-legal-vi-electra
precision: bf16=True  fp16=False  tf32=on


## 3. Nạp dữ liệu JSONL → HuggingFace Dataset

In [5]:
def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            # kiểm tra nhất quán
            assert len(obj["tokens"]) == len(obj["labels"]), f"Mismatch at id={obj.get('id')}"
            rows.append({
                "id": obj.get("id"),
                "tokens": obj["tokens"],
                "ner_tags": obj["labels"],  # tạm giữ tên chuỗi, sẽ map sang int
            })
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows   = load_jsonl(VAL_FILE)
test_rows  = load_jsonl(TEST_FILE)

print(f"train={len(train_rows)}  val={len(val_rows)}  test={len(test_rows)}")
print("ví dụ:", train_rows[0])

train=6403  val=1279  test=1288
ví dụ: {'id': 0, 'tokens': ['Khoản_1', 'Điều_46', 'quy_định', 'về', 'bảo_vệ', 'dữ_liệu', 'cá_nhân', 'áp_dụng', 'trong', 'lĩnh_vực', 'số', '.'], 'ner_tags': ['B-CLAUSE', 'B-ARTICLE', 'O', 'O', 'B-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'O', 'O', 'O', 'O', 'O']}


## 4. Xây dựng label space

Gom toàn bộ label từ cả 3 split để đảm bảo không bị miss entity type ở val/test. `O` luôn được đặt ở index 0.

In [6]:
label_counter = Counter()
for rows in (train_rows, val_rows, test_rows):
    for r in rows:
        label_counter.update(r["ner_tags"])

all_labels = sorted(label_counter.keys())
if "O" in all_labels:
    all_labels.remove("O")
LABEL_LIST = ["O"] + all_labels

label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABEL_LIST)

print(f"NUM_LABELS = {NUM_LABELS}")
print("Top label counts:")
for l, c in label_counter.most_common(15):
    print(f"  {l:<30} {c}")

# map string labels -> int
def encode_tags(rows):
    for r in rows:
        r["ner_tags"] = [label2id[t] for t in r["ner_tags"]]
    return rows

train_rows = encode_tags(train_rows)
val_rows   = encode_tags(val_rows)
test_rows  = encode_tags(test_rows)

raw_ds = DatasetDict({
    "train":      Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
    "test":       Dataset.from_list(test_rows),
})
raw_ds

NUM_LABELS = 125
Top label counts:
  O                              180210
  I-SANCTION                     38194
  I-LEGAL_ACTOR                  23804
  I-VIOLATION                    21504
  I-LAW                          21334
  I-CONDITION                    16880
  I-LEGAL_ACTION                 13556
  I-SYSTEM                       11151
  I-ATTACK                       9875
  B-OBLIGATION                   9506
  I-STANDARD                     9021
  B-LEGAL_ACTOR                  7786
  B-LEGAL_ACTION                 6722
  I-TIME                         5362
  I-DEPLOYMENT                   5269


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 6403
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1279
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1288
    })
})

## 5. Tokenizer + căn chỉnh label cho subword

Chiến lược chuẩn cho NER:
- First subtoken của mỗi word → giữ nguyên label.
- Các subtoken còn lại → gán `-100` (CrossEntropyLoss sẽ bỏ qua).
- Special tokens `[CLS]`, `[SEP]`, padding → `-100`.

→ Loss chỉ tính trên 1 đại diện cho mỗi word, đồng thời metric seqeval trả về đúng label theo word-level sau khi decode.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        aligned = []
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev_word:
                aligned.append(word_labels[wid])
            else:
                aligned.append(-100)  # subword sau của cùng 1 word
            prev_word = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = raw_ds.map(
    tokenize_and_align,
    batched=True,
    remove_columns=raw_ds["train"].column_names,
    desc="Tokenizing + aligning labels",
)
tokenized_ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizing + aligning labels:   0%|          | 0/6403 [00:00<?, ? examples/s]

Tokenizing + aligning labels:   0%|          | 0/1279 [00:00<?, ? examples/s]

Tokenizing + aligning labels:   0%|          | 0/1288 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6403
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1279
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1288
    })
})

## 6. Model — load base + thay head cho label mới

Vì head gốc của `NlpHUST/ner-vietnamese-electra-base` được train trên bộ tag khác, ta dùng `ignore_mismatched_sizes=True` để HF tự khởi tạo lại classifier layer với `NUM_LABELS` của mình, trong khi backbone vẫn giữ trọng số Electra đã train.

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(model.config.model_type, "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

model.safetensors:   0%|          | 0.00/532M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: NlpHUST/ner-vietnamese-electra-base
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
electra.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9]) vs model:torch.Size([125])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9, 768]) vs model:torch.Size([125, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


electra | params: 133.162877 M


## 7. Layer-wise Learning Rate Decay (LLRD)

Các layer càng gần input càng học feature tổng quát (cần LR nhỏ), các layer gần head cần LR lớn hơn. LLRD nhân LR theo hệ số `LLRD_DECAY^(N - layer_idx)`.

Head (classifier) dùng `CLASSIFIER_LR` riêng — cao hơn vì phải học từ đầu.

In [9]:
def build_llrd_optimizer(model, base_lr, classifier_lr, weight_decay, decay=0.9):
    no_decay = ("bias", "LayerNorm.weight")

    # Electra: model.electra.embeddings + model.electra.encoder.layer.{0..N-1}
    backbone = getattr(model, "electra", None) or getattr(model, "base_model")
    num_layers = len(backbone.encoder.layer)

    groups = []

    # embeddings = layer 0
    lr_emb = base_lr * (decay ** (num_layers + 1))
    emb_params = list(backbone.embeddings.named_parameters())
    groups.append({
        "params": [p for n, p in emb_params if not any(nd in n for nd in no_decay)],
        "lr": lr_emb, "weight_decay": weight_decay,
    })
    groups.append({
        "params": [p for n, p in emb_params if any(nd in n for nd in no_decay)],
        "lr": lr_emb, "weight_decay": 0.0,
    })

    # encoder layers
    for i, layer in enumerate(backbone.encoder.layer):
        lr_i = base_lr * (decay ** (num_layers - i))
        params = list(layer.named_parameters())
        groups.append({
            "params": [p for n, p in params if not any(nd in n for nd in no_decay)],
            "lr": lr_i, "weight_decay": weight_decay,
        })
        groups.append({
            "params": [p for n, p in params if any(nd in n for nd in no_decay)],
            "lr": lr_i, "weight_decay": 0.0,
        })

    # optional modules: embeddings_project (Electra base đôi khi có)
    if hasattr(backbone, "embeddings_project"):
        proj_params = list(backbone.embeddings_project.named_parameters())
        groups.append({
            "params": [p for n, p in proj_params if not any(nd in n for nd in no_decay)],
            "lr": lr_emb, "weight_decay": weight_decay,
        })
        groups.append({
            "params": [p for n, p in proj_params if any(nd in n for nd in no_decay)],
            "lr": lr_emb, "weight_decay": 0.0,
        })

    # classifier head — LR cao hơn
    head_params = []
    for name, p in model.named_parameters():
        if not name.startswith(("electra", "base_model")):
            head_params.append((name, p))
    groups.append({
        "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
        "lr": classifier_lr, "weight_decay": weight_decay,
    })
    groups.append({
        "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
        "lr": classifier_lr, "weight_decay": 0.0,
    })

    # bỏ group rỗng
    groups = [g for g in groups if len(g["params"]) > 0]
    return torch.optim.AdamW(groups, lr=base_lr)

print("LLRD optimizer builder sẵn sàng.")

LLRD optimizer builder sẵn sàng.


## 8. Metric — seqeval entity-level

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, lab_seq in zip(preds, labels):
        cur_t, cur_p = [], []
        for p_i, l_i in zip(pred_seq, lab_seq):
            if l_i == -100:
                continue
            cur_t.append(id2label[int(l_i)])
            cur_p.append(id2label[int(p_i)])
        true_labels.append(cur_t)
        true_preds.append(cur_p)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }

## 9. Training

In [11]:
import inspect

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding="longest",
    label_pad_token_id=-100,
)

# Xây dựng kwargs theo argument được hỗ trợ bởi phiên bản transformers hiện tại
# (tương thích cả bản cũ và bản v5+).
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
_tr_params = set(inspect.signature(Trainer.__init__).parameters.keys())

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    max_grad_norm=MAX_GRAD_NORM,
    label_smoothing_factor=LABEL_SMOOTHING,

    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,
    report_to="none",

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=SEED,
)

# --- evaluation strategy: tên arg đã đổi qua nhiều version ---
if "eval_strategy" in _ta_params:
    ta_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in _ta_params:
    ta_kwargs["evaluation_strategy"] = "epoch"

# --- warmup: v5 khuyến nghị warmup_steps; bản cũ dùng warmup_ratio ---
# Ước lượng tổng số update step để suy ra warmup_steps tương đương WARMUP_RATIO.
_steps_per_epoch = max(
    1,
    len(tokenized_ds["train"]) // (BATCH_SIZE_TRAIN * GRAD_ACCUM),
)
_total_steps = _steps_per_epoch * NUM_EPOCHS
_warmup_steps = int(_total_steps * WARMUP_RATIO)
if "warmup_steps" in _ta_params:
    ta_kwargs["warmup_steps"] = _warmup_steps
elif "warmup_ratio" in _ta_params:
    ta_kwargs["warmup_ratio"] = WARMUP_RATIO
print(f"warmup_steps ≈ {_warmup_steps} / {_total_steps} total steps")

# --- precision / misc chỉ có ở bản mới ---
if "data_seed" in _ta_params:
    ta_kwargs["data_seed"] = SEED
if "bf16" in _ta_params:
    ta_kwargs["bf16"] = USE_BF16
if "fp16" in _ta_params:
    ta_kwargs["fp16"] = USE_FP16
if "tf32" in _ta_params and HAS_CUDA:
    ta_kwargs["tf32"] = True

# logging_dir đã deprecate ở v5 — ưu tiên env var, fallback kwargs cho bản cũ.
os.environ["TENSORBOARD_LOGGING_DIR"] = str(LOG_DIR)
if "logging_dir" in _ta_params:
    # bản cũ vẫn chấp nhận, không warning ở các bản <5.0
    import transformers as _tf
    _major = int(_tf.__version__.split(".")[0])
    if _major < 5:
        ta_kwargs["logging_dir"] = str(LOG_DIR)

training_args = TrainingArguments(**ta_kwargs)

optimizer = build_llrd_optimizer(
    model,
    base_lr=BASE_LR,
    classifier_lr=CLASSIFIER_LR,
    weight_decay=WEIGHT_DECAY,
    decay=LLRD_DECAY,
)

# v5 đổi `tokenizer` → `processing_class`. Auto-detect.
tr_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),   # scheduler tự build theo warmup
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)
if "processing_class" in _tr_params:
    tr_kwargs["processing_class"] = tokenizer
elif "tokenizer" in _tr_params:
    tr_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**tr_kwargs)

print("=== Bắt đầu train ===")
train_result = trainer.train()
print(train_result.metrics)

warmup_steps ≈ 300 / 3000 total steps
=== Bắt đầu train ===


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,3.561698,3.472989,0.000000,0.000000,0.000000
2,2.668049,2.291112,0.284760,0.312669,0.298063
3,1.733917,1.364839,0.595106,0.688103,0.638234
4,1.267262,1.037206,0.844542,0.891576,0.867422
5,1.081239,0.933344,0.927449,0.948682,0.937945
6,0.965660,0.890820,0.945027,0.961801,0.953340
7,0.901956,0.866435,0.961710,0.972347,0.966999
8,0.881660,0.852902,0.977465,0.981865,0.979660
9,0.859558,0.845098,0.984493,0.988039,0.986263
10,0.846615,0.840819,0.987056,0.990611,0.988830


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

{'train_runtime': 261.6747, 'train_samples_per_second': 734.079, 'train_steps_per_second': 11.579, 'total_flos': 2.5120805884683756e+16, 'train_loss': 1.0941806100776093, 'epoch': 30.0}


## 10. Đánh giá trên tập test

In [12]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation:", val_metrics)

test_metrics = trainer.evaluate(tokenized_ds["test"])
print("Test:", test_metrics)

# classification_report chi tiết theo từng entity type
preds_output = trainer.predict(tokenized_ds["test"])
pred_ids = np.argmax(preds_output.predictions, axis=-1)
label_ids = preds_output.label_ids

y_true, y_pred = [], []
for p_seq, l_seq in zip(pred_ids, label_ids):
    t, p = [], []
    for p_i, l_i in zip(p_seq, l_seq):
        if l_i == -100:
            continue
        t.append(id2label[int(l_i)])
        p.append(id2label[int(p_i)])
    y_true.append(t)
    y_pred.append(p)

print("\n=== seqeval classification report (TEST) ===")
print(classification_report(y_true, y_pred, digits=4))

Validation: {'eval_loss': 0.8220987915992737, 'eval_precision': 0.996529116853066, 'eval_recall': 0.9970418006430868, 'eval_f1': 0.9967853928249968, 'eval_runtime': 1.4995, 'eval_samples_per_second': 852.961, 'eval_steps_per_second': 6.669, 'epoch': 30.0}
Test: {'eval_loss': 0.8178174495697021, 'eval_precision': 0.9960337768679631, 'eval_recall': 0.997565351101999, 'eval_f1': 0.9967989756722151, 'eval_runtime': 0.5783, 'eval_samples_per_second': 2227.185, 'eval_steps_per_second': 19.021, 'epoch': 30.0}

=== seqeval classification report (TEST) ===
                      precision    recall  f1-score   support

         APPLICATION     1.0000    1.0000    1.0000         3
             ARTICLE     1.0000    1.0000    1.0000       101
              ATTACK     1.0000    1.0000    1.0000       168
               AUDIT     0.8750    1.0000    0.9333         7
       AUTHORIZATION     0.7500    1.0000    0.8571         3
           BANDWIDTH     1.0000    1.0000    1.0000         1
         CE

## 11. Lưu model + label mapping

In [18]:
OUTPUT_DIR = PROJECT_ROOT / "/content/models" / "ner-legal-vi-electra"

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

print("Đã lưu model tại:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model tại: /content/models/ner-legal-vi-electra
['label_mapping.json', 'training_args.bin', 'model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']


In [31]:
import shutil
import os

# Create a zip file of the output directory
output_zip_path = "models.zip"
shutil.make_archive(output_zip_path.replace(".zip", ""), 'zip', OUTPUT_DIR)
print(f"Created {output_zip_path} which you can now download.")

# Optionally, you can list the current directory to see the created zip file
# !ls -lh

Created models.zip which you can now download.


## 12. Demo inference

Gom các subword về word-level và merge liên tiếp `B-X` / `I-X` thành entity cuối cùng.

In [ ]:
"""Inference pipeline hoàn chỉnh — v2
  1. word_tokenize (underthesea + merge_tokens) — khớp data train
  2. NER model predict 60 entity types
  3. Rule-based bắt 5 Structural types (PART/CHAPTER/SECTION/POINT/APPENDIX)
  4. Post-process: filter score, dedup, sort by position
  5. Fix inline:
       yêu_cầu → CONDITION
       TELECOM → LEGAL_ACTOR (subclass)
       DATA_TYPE whitelist
  6. [NEW] Rule-based LOCATION + SYSTEM whitelist catch
  7. [NEW] RELATION extractor → triples (ACTOR, relation, OBJECT)
  8. Format output: raw entities + graph-ready nodes + triples
"""

import re
import json
import torch
import numpy as np
from underthesea import word_tokenize as _ut
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ── Config ─────────────────────────────────────────────────────
MODEL_DIR       = str(OUTPUT_DIR)
MAX_LENGTH      = 512
SCORE_THRESHOLD = 0.5

# ── Load model ─────────────────────────────────────────────────
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).eval()
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mdl.to(device)
label_set = json.load(open(f"{MODEL_DIR}/label_mapping.json", encoding="utf-8"))
id2label  = {int(k): v for k, v in label_set["id2label"].items()}


# ================================================================
# BƯỚC 1: Tokenization
# ================================================================
def merge_tokens(tokens: list[str]) -> list[str]:
    TIME_UNITS   = {'giờ','ngày','tháng','năm','tuần','phút','giây'}
    MONEY_UNITS  = {'triệu','tỷ','nghìn','trăm'}
    TECH_UNITS   = {'Gbps','Mbps','MB','GB','TB','KB','GHz','MHz','ms','bps'}
    CURRENCIES   = {'USD','EUR','VND','GBP','JPY'}
    STRUCT_WORDS = {'Điều','Khoản','Điểm','Mục','Chương','Phần'}
    STANDARDS    = {'ISO','IEC','NIST','OWASP','CWE','CVE','CVSS','PCI','ETSI','TCVN','QCVN'}

    result, i, n = [], 0, len(tokens)
    while i < n:
        t = tokens[i]
        if re.match(r'^\d[\d,\.]*$', t) and i+1 < n and tokens[i+1] in TIME_UNITS:
            m = t + '_' + tokens[i+1]; i += 2
            if i < n and tokens[i] == 'làm_việc': m += '_làm_việc'; i += 1
            result.append(m); continue
        if re.match(r'^\d[\d,\.]*$', t) and i+1 < n and tokens[i+1] in MONEY_UNITS:
            m = t + '_' + tokens[i+1]; i += 2
            if i < n and tokens[i] == 'đồng': m += '_đồng'; i += 1
            result.append(m); continue
        if re.match(r'^\d[\d,\.]*$', t) and i+1 < n and tokens[i+1] == '%':
            result.append(t + '%'); i += 2; continue
        if re.match(r'^\d[\d,\.]*$', t) and i+1 < n and tokens[i+1] in TECH_UNITS:
            result.append(t + '_' + tokens[i+1]); i += 2; continue
        if re.match(r'^\d[\d,\.]*$', t) and i+1 < n and tokens[i+1] in CURRENCIES:
            result.append(t + '_' + tokens[i+1]); i += 2; continue
        if re.match(r'^\d+$', t) and i+1 < n and tokens[i+1] == 'lần':
            result.append(t + '_lần'); i += 2; continue
        if t in STRUCT_WORDS and i+1 < n and re.match(r'^\d+$|^[a-zđ]$', tokens[i+1]):
            result.append(t + '_' + tokens[i+1]); i += 2; continue
        if t in STANDARDS and i+1 < n:
            nxt = tokens[i+1]
            if re.match(r'^\d[\d\-\.]*$|^SP$|^DSS$|^MASVS$', nxt):
                m = t + '_' + nxt; i += 2
                if i < n and tokens[i] == ':' and i+1 < n and re.match(r'^\d+$', tokens[i+1]):
                    m += ':' + tokens[i+1]; i += 2
                result.append(m); continue
            if nxt == '/' and i+2 < n and tokens[i+2] in STANDARDS:
                m = t + '/' + tokens[i+2]; i += 3
                if i < n and re.match(r'^\d[\d\-\.]*$', tokens[i]):
                    m += '_' + tokens[i]; i += 1
                result.append(m); continue
        if t == 'CVE' and i+1 < n and tokens[i+1] == '-' and i+2 < n:
            if re.match(r'^\d{4}$', tokens[i+2]):
                m = 'CVE-' + tokens[i+2]; i += 3
                if i < n and tokens[i] == '-' and i+1 < n:
                    m += '-' + tokens[i+1]; i += 2
                result.append(m); continue
        if t == 'Tier' and i+1 < n and re.match(r'^(I{1,3}|IV|V|\d+)$', tokens[i+1]):
            result.append('Tier_' + tokens[i+1]); i += 2; continue
        if t == 'cấp' and i+1 < n and tokens[i+1] in ('độ','_độ') and i+2 < n:
            if re.match(r'^\d+$', tokens[i+2]):
                result.append('cấp_độ_' + tokens[i+2]); i += 3; continue
        result.append(t); i += 1
    return result


def word_tokenize(text: str) -> list[str]:
    return merge_tokens(_ut(text, format='text').split())


# ================================================================
# BƯỚC 2: NER Model predict
# ================================================================
@torch.no_grad()
def _ner_predict(text: str) -> tuple[list[str], list[dict]]:
    words = word_tokenize(text)
    enc = tok(
        words, is_split_into_words=True,
        return_tensors="pt", truncation=True, max_length=MAX_LENGTH,
    ).to(device)

    logits   = mdl(**enc).logits[0]
    probs    = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(-1)
    word_ids = enc.word_ids(0)

    word_label = [None] * len(words)
    word_score = [0.0]  * len(words)
    seen = set()
    for j, wid in enumerate(word_ids):
        if wid is None or wid in seen: continue
        seen.add(wid)
        word_label[wid] = id2label[int(pred_ids[j])]
        word_score[wid] = float(probs[j, pred_ids[j]])

    entities, cur = [], None

    def flush():
        if cur:
            entities.append({
                "type":   cur["type"],
                "text":   " ".join(cur["tokens"]).replace("_", " "),
                "score":  round(sum(cur["scores"]) / len(cur["scores"]), 4),
                "start":  cur["start"],
                "end":    cur["end"],
                "source": "model",
            })

    for i, (w, lab, s) in enumerate(zip(words, word_label, word_score)):
        if lab is None or lab == "O":
            flush(); cur = None; continue
        prefix, _, etype = lab.partition("-")
        if prefix == "B" or cur is None or cur["type"] != etype:
            flush()
            cur = {"type": etype, "tokens": [w], "scores": [s], "start": i, "end": i+1}
        else:
            cur["tokens"].append(w); cur["scores"].append(s); cur["end"] = i+1
    flush()
    return words, entities


# ================================================================
# BƯỚC 3: Rule-based Structural types
# ================================================================
STRUCTURAL_PATTERNS = [
    (r'(?<!\w)(Phần\s+(?:[IVX]+|\d+))(?!\w)',             'PART'),
    (r'(?<!\w)(Chương\s+(?:[IVX]+|\d+))(?!\w)',            'CHAPTER'),
    (r'(?<!\w)(Mục\s+\d+)(?!\w)',                          'SECTION'),
    (r'(?<!\w)([Đđ]iểm\s+[a-zđ])(?!\w)',                  'POINT'),
    (r'(?<!\w)(Phụ\s+lục\s+(?:[IVX]+|[A-Z]|\d+))(?!\w)', 'APPENDIX'),
]

def _rule_structural(text: str) -> list[dict]:
    found = []
    for pattern, etype in STRUCTURAL_PATTERNS:
        for m in re.finditer(pattern, text, re.IGNORECASE):
            found.append({
                "type": etype, "text": m.group(1).strip(),
                "score": 1.0, "char_start": m.start(1),
                "char_end": m.end(1), "source": "rule",
            })
    return found


# ================================================================
# BƯỚC 3b: [NEW] Rule-based LOCATION + SYSTEM whitelist
# ================================================================

# ── LOCATION whitelist (longest-match first) ───────────────────
_LOCATION_PHRASES = sorted([
    'ngoài lãnh thổ Việt Nam', 'lãnh thổ Việt Nam',
    'trong lãnh thổ Việt Nam', 'Việt Nam',
    'nước ngoài', 'trong nước', 'quốc tế',
    'Hà Nội', 'TP. Hồ Chí Minh', 'TP.HCM', 'Hồ Chí Minh',
    'Đà Nẵng', 'Cần Thơ', 'Hải Phòng',
], key=len, reverse=True)

def _rule_location(text: str) -> list[dict]:
    found = []
    used  = []
    text_lower = text.lower()
    for phrase in _LOCATION_PHRASES:
        search = phrase.lower()
        idx = 0
        while True:
            pos = text_lower.find(search, idx)
            if pos == -1: break
            end = pos + len(phrase)
            if not any(s <= pos < e or s < end <= e for s, e in used):
                found.append({
                    "type": "LOCATION", "text": text[pos:end],
                    "score": 1.0, "char_start": pos,
                    "char_end": end, "source": "rule",
                })
                used.append((pos, end))
            idx = pos + 1
    return found


# ── SYSTEM whitelist ───────────────────────────────────────────
_SYSTEM_PHRASES = sorted([
    'hệ thống thông tin', 'hệ thống máy chủ', 'hệ thống mạng',
    'hệ thống lõi', 'hệ thống giám sát', 'hệ thống phát hiện xâm nhập',
    'cơ sở dữ liệu', 'máy chủ', 'hạ tầng thông tin', 'hạ tầng số',
    'hạ tầng kỹ thuật số', 'nền tảng số', 'điện toán đám mây',
    'đám mây', 'phần mềm độc hại', 'tường lửa', 'phần mềm',
    'ứng dụng di động', 'thiết bị đầu cuối',
], key=len, reverse=True)

def _rule_system(text: str, existing_entities: list[dict]) -> list[dict]:
    """Bắt SYSTEM entities bị bỏ sót bởi model."""
    existing_texts = {
        e['text'].lower().replace('_', ' ').strip()
        for e in existing_entities
        if e['type'] == 'SYSTEM'
    }
    found  = []
    used   = [(e.get('char_start', -1), e.get('char_end', -1)) for e in existing_entities]
    text_lower = text.lower()

    for phrase in _SYSTEM_PHRASES:
        if phrase in existing_texts: continue
        idx = 0
        while True:
            pos = text_lower.find(phrase, idx)
            if pos == -1: break
            end = pos + len(phrase)
            if not any(s != -1 and s <= pos < e for s, e in used):
                found.append({
                    "type": "SYSTEM", "text": text[pos:end],
                    "score": 0.9, "char_start": pos,
                    "char_end": end, "source": "rule",
                })
                used.append((pos, end))
                existing_texts.add(phrase)
            idx = pos + 1
    return found


# ================================================================
# BƯỚC 4: Post-process — filter + dedup + sort
# ================================================================
def _clean_entities(model_ents, rule_ents, threshold):
    model_ents = [e for e in model_ents if e["score"] >= threshold]
    structural = {'PART', 'CHAPTER', 'SECTION', 'POINT', 'APPENDIX'}
    model_ents = [e for e in model_ents if e["type"] not in structural]
    all_ents   = model_ents + rule_ents
    all_ents.sort(key=lambda e: e.get("start", e.get("char_start", 0)))
    seen, deduped = {}, []
    for e in all_ents:
        key = (e["type"], e["text"].lower().strip())
        if key not in seen:
            seen[key] = e; deduped.append(e)
        elif e["score"] > seen[key]["score"]:
            deduped[deduped.index(seen[key])] = e; seen[key] = e
    return deduped


# ================================================================
# BƯỚC 5: Fix inline
# ================================================================

_COND_TRIGGERS = {
    'khi', 'nếu', 'trường_hợp', 'khi_có', 'khi có',
    'trong_trường_hợp', 'trong trường hợp',
}

def _fix_yeu_cau(entities: list[dict], words: list[str]) -> list[dict]:
    words_lower = [w.lower() for w in words]
    result = []
    for e in entities:
        if e['type'] == 'OBLIGATION' and e['text'].lower().strip() in ('yêu cầu', 'yêu_cầu'):
            start = e.get('start', 0)
            prev  = set(words_lower[max(0, start-3):start])
            after = set(words_lower[e.get('end', start+1):e.get('end', start+1)+2])
            if prev & _COND_TRIGGERS or start <= 1 or 'bằng_văn_bản' in after or 'bằng' in after:
                result.append({**e, 'type': 'CONDITION'}); continue
        result.append(e)
    return result


def _fix_telecom(entities: list[dict]) -> list[dict]:
    return [
        {**e, 'type': 'LEGAL_ACTOR', 'subclass': 'TELECOM_OPERATOR'}
        if e['type'] == 'TELECOM_OPERATOR' else e
        for e in entities
    ]


_DATA_WHITELIST = {
    'dữ liệu cá nhân', 'dữ liệu cá nhân nhạy cảm', 'dữ liệu sinh trắc học',
    'dữ liệu sức khỏe', 'dữ liệu tài chính', 'dữ liệu y tế', 'dữ liệu nhạy cảm',
    'dữ liệu hành vi cá nhân', 'dữ liệu di truyền', 'dữ liệu vị trí',
    'dữ liệu người sử dụng', 'dữ liệu người dùng', 'dữ liệu thuê bao',
    'nhật ký truy cập', 'nhật ký hoạt động', 'thông tin cá nhân',
    'thông tin người dùng', 'thông tin thuê bao', 'thông tin đăng ký',
    'dữ liệu bí mật nhà nước', 'thông tin bảo mật',
}

def _fix_data_type(entities: list[dict], text: str) -> list[dict]:
    result = list(entities)
    text_lower = text.lower()
    existing = {e['text'].lower().replace('_', ' ').strip() for e in result}

    for e in result:
        if e['type'] in ('DATA_TYPE', 'DATA'):
            if e['text'].lower().replace('_', ' ').strip() in _DATA_WHITELIST:
                e['score'] = max(e['score'], 0.85)

    for phrase in sorted(_DATA_WHITELIST, key=len, reverse=True):
        if phrase not in existing and phrase in text_lower:
            idx = text_lower.find(phrase)
            result.append({
                'type': 'DATA_TYPE', 'text': text[idx:idx+len(phrase)],
                'score': 0.85, 'start': -1, 'end': -1, 'source': 'whitelist',
            })
            existing.add(phrase)
    return result


# ================================================================
# BƯỚC 7: [NEW] RELATION extractor
# ================================================================

# Normative → relation type mapping
_NORM_TO_REL = {
    'OBLIGATION': 'OBLIGATES',
    'PROHIBITION': 'PROHIBITS',
    'PERMISSION':  'PERMITS',
}

# Action verbs → relation label
_ACTION_TO_REL = {
    'lưu trữ':     'STORES',
    'cung cấp':    'PROVIDES',
    'xử lý':       'PROCESSES',
    'thu thập':    'COLLECTS',
    'triển khai':  'DEPLOYS',
    'thực hiện':   'PERFORMS',
    'áp dụng':     'APPLIES',
    'bảo đảm':     'ENSURES',
    'bảo vệ':      'PROTECTS',
    'kiểm tra':    'INSPECTS',
    'đánh giá':    'EVALUATES',
    'xử phạt':     'PENALIZES',
    'truy cứu':    'PROSECUTES',
    'đình chỉ':    'SUSPENDS',
    'chuyển giao': 'TRANSFERS',
    'chia sẻ':     'SHARES',
    'xóa':         'DELETES',
    'mã hóa':      'ENCRYPTS',
    'báo cáo':     'REPORTS',
    'thông báo':   'NOTIFIES',
}

# Entity types that can be subject / object in a triple
_ACTOR_TYPES   = {'LEGAL_ACTOR', 'LEGAL_ENTITY', 'AUTHORITY', 'GOVERNMENT'}
_OBJECT_TYPES  = {'DATA_TYPE', 'DATA', 'SYSTEM', 'SERVICE', 'LEGAL_OBJECT',
                  'LEGAL_CONCEPT', 'RIGHT', 'SANCTION'}
_NORM_TYPES    = {'OBLIGATION', 'PERMISSION', 'PROHIBITION'}
_ACTION_TYPES  = {'LEGAL_ACTION'}


def _nearest(entities: list[dict], pos: int, allowed_types: set,
             direction: str = 'left', window: int = 5) -> dict | None:
    """Tìm entity gần nhất trong allowed_types theo hướng left/right."""
    candidates = []
    for e in entities:
        e_pos = e.get('start', e.get('char_start', 0))
        if e['type'] not in allowed_types: continue
        if direction == 'left' and e_pos < pos:
            candidates.append((pos - e_pos, e))
        elif direction == 'right' and e_pos > pos:
            candidates.append((e_pos - pos, e))
    if not candidates: return None
    candidates.sort(key=lambda x: x[0])
    dist, best = candidates[0]
    return best if dist <= window else None


def extract_relations(entities: list[dict]) -> list[dict]:
    """
    Strategy:
      A. Normative triple: ACTOR --[NORM]--> ACTION
         e.g. "doanh nghiệp [phải] [lưu trữ] dữ liệu"
         → (doanh nghiệp, OBLIGATES, lưu trữ)

      B. Action triple: ACTOR --[ACTION]--> OBJECT
         e.g. "doanh nghiệp [lưu trữ] dữ liệu cá nhân"
         → (doanh nghiệp, STORES, dữ liệu cá nhân)

      C. Norm-Object triple: ACTOR --[NORM]-- OBJECT (khi không có ACTION rõ)
         e.g. "doanh nghiệp [không được] [cung cấp] dữ liệu"
         → (doanh nghiệp, PROHIBITS, dữ liệu người sử dụng)
    """
    triples = []
    seen_triples = set()

    def add_triple(subj, rel, obj, confidence):
        key = (subj['text'].lower(), rel, obj['text'].lower())
        if key in seen_triples: return
        seen_triples.add(key)
        triples.append({
            "subject":    {"type": subj["type"], "text": subj["text"]},
            "relation":   rel,
            "object":     {"type": obj["type"],  "text": obj["text"]},
            "confidence": round(confidence, 3),
        })

    # Process each ACTION entity
    for action_ent in entities:
        if action_ent['type'] not in _ACTION_TYPES: continue
        action_pos = action_ent.get('start', action_ent.get('char_start', 0))
        action_text = action_ent['text'].lower().replace('_', ' ').strip()

        # Map to relation label
        rel_label = _ACTION_TO_REL.get(action_text, 'RELATES_TO')

        # Find nearest ACTOR to the left (window=8 words)
        actor = _nearest(entities, action_pos, _ACTOR_TYPES, 'left', window=8)

        # Find nearest OBJECT to the right (window=6 words)
        obj = _nearest(entities, action_pos, _OBJECT_TYPES, 'right', window=6)

        if actor and obj:
            conf = (actor['score'] + action_ent['score'] + obj['score']) / 3
            add_triple(actor, rel_label, obj, conf)

        # Also check if there's a normative modal between actor and action
        if actor:
            norm = _nearest(entities, action_pos, _NORM_TYPES, 'left', window=3)
            if norm:
                norm_rel = _NORM_TO_REL.get(norm['type'], 'CONSTRAINS')
                if obj:
                    # Full triple: actor -[norm+action]-> obj
                    compound_rel = f"{norm_rel}_{rel_label}"
                    conf = (actor['score'] + norm['score'] + action_ent['score'] + obj['score']) / 4
                    add_triple(actor, compound_rel, obj, conf)

    # Process SANCTION entities
    for sanction_ent in entities:
        if sanction_ent['type'] != 'SANCTION': continue
        sanction_pos = sanction_ent.get('start', sanction_ent.get('char_start', 0))
        actor = _nearest(entities, sanction_pos, _ACTOR_TYPES, 'left', window=10)
        if actor:
            conf = (actor['score'] + sanction_ent['score']) / 2
            add_triple(actor, 'SUBJECT_TO_SANCTION', sanction_ent, conf)

    # LAW / ARTICLE references
    _REF_TYPES = {'LAW', 'ARTICLE', 'CLAUSE'}
    for ref_ent in entities:
        if ref_ent['type'] not in _REF_TYPES: continue
        ref_pos = ref_ent.get('start', ref_ent.get('char_start', 0))
        actor = _nearest(entities, ref_pos, _ACTOR_TYPES, 'right', window=6)
        if actor:
            conf = (ref_ent['score'] + actor['score']) / 2
            add_triple(ref_ent, 'GOVERNS', actor, conf)

    return sorted(triples, key=lambda t: -t['confidence'])


# ================================================================
# BƯỚC 8: Format output
# ================================================================
def _make_node_id(etype: str, text: str) -> str:
    clean = re.sub(r'\s+', '_', text.lower().strip())
    clean = re.sub(r'[^\w_]', '', clean)
    return f"{etype.lower()}_{clean[:50]}"


def _format_output(text: str, entities: list[dict],
                   law_ref: str = None) -> dict:
    nodes, seen_ids = [], set()
    for e in entities:
        nid = _make_node_id(e["type"], e["text"])
        if nid in seen_ids: continue
        seen_ids.add(nid)
        node = {
            "id":     nid,
            "type":   e["type"],
            "label":  e["text"],
            "score":  e["score"],
            "source": e.get("source", "model"),
        }
        if e.get("subclass"): node["subclass"] = e["subclass"]
        if law_ref: node["law_ref"] = law_ref
        nodes.append(node)

    type_counts = {}
    for e in entities:
        type_counts[e["type"]] = type_counts.get(e["type"], 0) + 1

    triples = extract_relations(entities)

    return {
        "text":    text,
        "raw":     entities,
        "nodes":   nodes,
        "triples": triples,
        "summary": {
            "total_entities": len(entities),
            "unique_nodes":   len(nodes),
            "total_triples":  len(triples),
            "by_type":        dict(sorted(type_counts.items(), key=lambda x: -x[1])),
        },
    }


# ================================================================
# ENTRY POINT
# ================================================================
def predict_entities(text: str,
                     score_threshold: float = SCORE_THRESHOLD,
                     law_ref: str = None) -> dict:
    words, model_ents = _ner_predict(text)

    # Rule-based enrichment
    rule_ents  = _rule_structural(text)
    rule_ents += _rule_location(text)

    entities = _clean_entities(model_ents, rule_ents, score_threshold)

    # Inline fixes
    entities = _fix_yeu_cau(entities, words)
    entities = _fix_telecom(entities)
    entities = _fix_data_type(entities, text)

    # [NEW] SYSTEM whitelist catch (after initial entity list is stable)
    entities += _rule_system(text, entities)

    # Final dedup
    seen, deduped = set(), []
    for e in sorted(entities, key=lambda e: -e['score']):
        key = (e["type"], e["text"].lower().replace('_', ' ').strip())
        if key not in seen:
            seen.add(key); deduped.append(e)

    # Sort by position for output
    deduped.sort(key=lambda e: e.get('start', e.get('char_start', 999)))

    return _format_output(text, deduped, law_ref=law_ref)


# ================================================================
# PRINT HELPER
# ================================================================
def print_result(result: dict, show_nodes: bool = False, show_triples: bool = True):
    print(f"\n{'─'*72}")
    print(f"TEXT: {result['text'][:80]}{'...' if len(result['text'])>80 else ''}")
    print(f"{'─'*72}")

    by_type = {}
    for e in result["raw"]:
        by_type.setdefault(e["type"], []).append(e)

    for etype, ents in sorted(by_type.items()):
        for e in ents:
            src = "📐" if e.get("source") in ("rule", "whitelist") else "🤖"
            sub = f" [{e['subclass']}]" if e.get("subclass") else ""
            print(f"  {src} {etype:<22}{sub} '{e['text']}'  ({e['score']:.3f})")

    if show_triples and result.get("triples"):
        print(f"\n  🔗 Relations ({len(result['triples'])}):")
        for t in result["triples"]:
            conf = t['confidence']
            bar  = '█' * int(conf * 10) + '░' * (10 - int(conf * 10))
            print(f"     ({t['subject']['text']})  ──[{t['relation']}]──▶  ({t['object']['text']})  {bar} {conf:.2f}")

    if show_nodes:
        print(f"\n  📦 Nodes ({result['summary']['unique_nodes']}):")
        for node in result["nodes"]:
            print(f"     {node['id']}")

    print(f"\n  📊 {result['summary']['by_type']}")
    print(f"  🔗 Triples extracted: {result['summary']['total_triples']}")


# ================================================================
# RUN EXAMPLES
# ================================================================
examples = [
    ("Theo khoản 2 Điều 15 của Luật An ninh mạng, doanh nghiệp viễn thông và doanh nghiệp cung cấp dịch vụ internet có trách nhiệm lưu trữ dữ liệu cá nhân của người sử dụng tại Việt Nam, bao gồm thông tin đăng ký, thông tin truy cập và nhật ký hoạt động, trong thời gian tối thiểu 24 tháng kể từ thời điểm phát sinh dữ liệu, nhằm phục vụ công tác quản lý nhà nước về an ninh mạng.",
     "Luật An ninh mạng 2025"),

    ("Khi có yêu cầu bằng văn bản của cơ quan có thẩm quyền, doanh nghiệp viễn thông phải cung cấp dữ liệu người sử dụng, bao gồm dữ liệu cá nhân và nhật ký truy cập, cho Bộ Công an hoặc Bộ Thông tin và Truyền thông để phục vụ công tác điều tra, thanh tra và xử lý vi phạm pháp luật về an ninh mạng.",
     "Luật An ninh mạng 2025"),

    ("Việc cung cấp dữ liệu quy định tại Điều 15 phải được thực hiện theo đúng trình tự, thủ tục do pháp luật quy định và chỉ áp dụng trong trường hợp cần thiết nhằm bảo vệ an ninh quốc gia, trật tự an toàn xã hội, đồng thời phải bảo đảm quyền và lợi ích hợp pháp của người sử dụng, trừ trường hợp pháp luật có quy định khác.",
     "Luật An ninh mạng 2025"),

    ("Doanh nghiệp có trách nhiệm triển khai hệ thống máy chủ đặt tại lãnh thổ Việt Nam, áp dụng các biện pháp kỹ thuật cần thiết để bảo đảm an toàn thông tin, phòng chống truy cập trái phép, phát hiện và ngăn chặn các hành vi tấn công mạng, đồng thời thực hiện kiểm tra, đánh giá định kỳ đối với hệ thống thông tin và cơ sở dữ liệu.",
     "Luật An ninh mạng 2025"),

    ("Trường hợp doanh nghiệp không thực hiện hoặc thực hiện không đầy đủ các nghĩa vụ quy định tại các điều khoản nêu trên, tùy theo tính chất và mức độ vi phạm, sẽ bị xử lý theo quy định của pháp luật, bao gồm xử phạt vi phạm hành chính, đình chỉ hoạt động cung cấp dịch vụ hoặc truy cứu trách nhiệm hình sự theo quy định hiện hành.",
     "Luật An ninh mạng 2025"),

    # ===================== HARD CASES =====================

    ("Trong trường hợp doanh nghiệp viễn thông không thể lưu trữ dữ liệu người sử dụng tại Việt Nam do sự cố kỹ thuật bất khả kháng, thì phải thông báo bằng văn bản cho Bộ Thông tin và Truyền thông trong thời hạn không quá 24 giờ kể từ thời điểm xảy ra sự cố; đồng thời, doanh nghiệp chỉ được phép chuyển dữ liệu ra nước ngoài sau khi đã có sự chấp thuận bằng văn bản của cơ quan này, trừ trường hợp pháp luật có quy định khác.",
     "Luật An ninh mạng 2025"),

    ("Bộ Công an, Bộ Thông tin và Truyền thông và các cơ quan có thẩm quyền khác có trách nhiệm phối hợp với doanh nghiệp cung cấp dịch vụ internet trong việc kiểm tra, giám sát và yêu cầu cung cấp dữ liệu khi cần thiết nhằm bảo đảm an ninh quốc gia, trật tự an toàn xã hội và phòng, chống tội phạm công nghệ cao.",
     "Luật An ninh mạng 2025"),

    ("Doanh nghiệp viễn thông không được phép cung cấp, chia sẻ hoặc chuyển giao dữ liệu cá nhân của người sử dụng cho bất kỳ tổ chức, cá nhân nào khác, trừ trường hợp có sự đồng ý rõ ràng của người sử dụng hoặc theo yêu cầu bằng văn bản của cơ quan nhà nước có thẩm quyền theo quy định của pháp luật.",
     "Luật An ninh mạng 2025"),

    ("Doanh nghiệp có trách nhiệm thiết lập, vận hành và duy trì hệ thống thông tin đáp ứng các tiêu chuẩn an toàn thông tin theo quy định của pháp luật, bao gồm việc áp dụng các biện pháp mã hóa dữ liệu, kiểm soát truy cập, phát hiện và ngăn chặn hành vi truy cập trái phép, đồng thời thực hiện đánh giá định kỳ và báo cáo kết quả cho cơ quan quản lý nhà nước có thẩm quyền.",
     "Luật An ninh mạng 2025"),

    ("Trường hợp doanh nghiệp không thực hiện nghĩa vụ lưu trữ dữ liệu theo quy định tại Điều 15 hoặc thực hiện không đầy đủ dẫn đến hậu quả nghiêm trọng, thì tùy theo tính chất và mức độ vi phạm, có thể bị xử phạt vi phạm hành chính, đình chỉ hoạt động cung cấp dịch vụ trong thời hạn từ 01 tháng đến 12 tháng, hoặc bị truy cứu trách nhiệm hình sự theo quy định của pháp luật.",
     "Luật An ninh mạng 2025"),
]

for text, law in examples:
    result = predict_entities(text, score_threshold=0.5, law_ref=law)
    print_result(result, show_nodes=False, show_triples=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────────────────
TEXT: Theo khoản 2 Điều 15 của Luật An ninh mạng, doanh nghiệp viễn thông và doanh ngh...
────────────────────────────────────────────────────────────────────────
  🤖 ARTICLE                'Điều 15'  (0.951)
  🤖 CLAUSE                 'khoản 2'  (0.944)
  📐 DATA_TYPE              'nhật ký hoạt động'  (0.850)
  📐 DATA_TYPE              'thông tin đăng ký'  (0.850)
  📐 DATA_TYPE              'dữ liệu cá nhân'  (0.850)
  🤖 LAW                    'Luật An ninh mạng'  (0.929)
  🤖 LEGAL_ACTION           'lưu trữ dữ liệu'  (0.911)
  🤖 LEGAL_ACTOR            'doanh nghiệp viễn thông'  (0.888)
  🤖 LEGAL_ACTOR            'doanh nghiệp cung cấp dịch vụ internet'  (0.931)
  📐 LOCATION               'Việt Nam'  (1.000)
  🤖 OBLIGATION             'có trách nhiệm'  (0.905)
  🤖 SCOPE                  'công tác quản lý nhà nước về an ninh mạng'  (0.746)
  🤖 TIME                   'tối thiểu 24 tháng'  (0.931)

  🔗 Relations (3):

## 13. (Tuỳ chọn) Export nhanh — tích hợp vào `backend/graph_rag/ner_extractor.py`

> ⚠️ **Lưu ý quan trọng:** model được train trên data đã word-tokenize bằng `pyvi`
> (token có dạng `Bộ_Công_an`, `an_ninh_mạng`, ...). Inference **bắt buộc** phải
> chạy đúng pipeline `word_tokenize → is_split_into_words=True → merge BIO`,
> nếu không entity sẽ lệch hoàn toàn (xem cell demo §12).

```python
import torch
from pyvi import ViTokenizer
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_DIR = "models/ner-legal-vi-electra"
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).eval()
id2label = mdl.config.id2label

@torch.no_grad()
def extract_entities(text: str):
    words = ViTokenizer.tokenize(text).split()
    enc = tok(words, is_split_into_words=True, return_tensors="pt", truncation=True)
    pred_ids = mdl(**enc).logits[0].argmax(-1).tolist()
    word_ids = enc.word_ids(0)

    word_label = [None] * len(words)
    seen = set()
    for j, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        word_label[wid] = id2label[pred_ids[j]]

    entities, cur = [], None
    for w, lab in zip(words, word_label):
        if lab in (None, "O"):
            if cur: entities.append(cur); cur = None
            continue
        prefix, _, etype = lab.partition("-")
        if prefix == "B" or cur is None or cur["type"] != etype:
            if cur: entities.append(cur)
            cur = {"type": etype, "text": w.replace("_", " ")}
        else:
            cur["text"] += " " + w.replace("_", " ")
    if cur: entities.append(cur)
    return entities
```